In [1]:
# Import des librairies nécessaires
import pandas as pd
import numpy as np
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import folium
from tqdm import tqdm  # pour voir la progression

fichier = "/home/fatma/Documents/machinelearning/capstone_project/data/test_prepared.csv"

# Charger les données
df = pd.read_csv(fichier)

# Afficher les premières lignes
print("Shape:", df.shape)
df.head()

Shape: (289, 12)


,id,titre,surface_m2,nb_chambres,nb_salons,nb_sdb,quartier,description,caracteristiques,source,date_publication,quartier_encoded
0,528,فرصة سمعة للبيع في كارفور,150.0,8.0,5.0,2.0,arafat,سمعة مؤلفة من طابق أرضي وطابق أول وأستديوه ومح...,1 balcon(s) | Taille rue: 30.00 | Proche de: ب...,voursa.com,2025-04-09,1.676908e+06
1,1296,دار فدايه 19,180.0,5.0,1.0,2.0,arafat,حد دوره تلفنل اعل واتساب 26678975\nللاتصال 361...,Titre foncier | 1 balcon(s) | Taille rue: 20.0...,voursa.com,2025-08-13,1.676908e+06
2,1286,منزل جديدة ملح سكتير 2 مافات اندخلت,150.0,3.0,2.0,2.0,toujounine,السلام عليكم \nدغر جديدة للبيـــــــــــــــــ...,Titre foncier | 1 balcon(s) | Taille rue: 15.0...,voursa.com,2025-04-17,1.571960e+06
3,1021,دار كبيره فداي 11 دخلاني,180.0,4.0,2.0,2.0,arafat,فيه اربع بيوت ومرافق وفيه بوتيك مكري فاتح اعل ...,Titre foncier | 1 balcon(s) | Taille rue: 30.0...,voursa.com,2025-10-28,1.676908e+06
4,64,دار البركه اعل كدروه,180.0,4.0,2.0,2.0,teyarett,سعر 18 مليون واحذ العنكار حد ايدور شي تلفنل 26...,Titre foncier | 1 balcon(s) | Taille rue: 30.0...,voursa.com,2025-11-07,3.330686e+06


In [2]:
# Voir tous les quartiers uniques
print("Quartiers uniques avant nettoyage:")
print(df['quartier'].unique())
print(f"\nNombre de quartiers uniques: {df['quartier'].nunique()}")

Quartiers uniques avant nettoyage:
['arafat' 'toujounine' 'teyarett' 'tevragh zeina' 'dar naim' 'sebkha'
 'riyadh' 'ksar']

Nombre de quartiers uniques: 8


# Étape 1 : Géocodage des quartiers

In [3]:
# Initialiser le géocodeur Nominatim
geolocator = Nominatim(user_agent="capstone_equipe_predictors")

# Créer un rate limiter pour respecter les 1 requête/seconde
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Dictionnaire pour stocker les résultats
coord_quartiers = {}

In [4]:
# Test sur Tevragh Zeina
test_quartier = "Tevragh Zeina"
print(f"Recherche de '{test_quartier}, Nouakchott, Mauritanie'...")

try:
    location = geolocator.geocode(f"{test_quartier}, Nouakchott, Mauritanie")
    if location:
        print(f"✅ Trouvé !")
        print(f"   Latitude: {location.latitude}")
        print(f"   Longitude: {location.longitude}")
        print(f"   Adresse complète: {location.address}")
    else:
        print(f"❌ Non trouvé")
except Exception as e:
    print(f"❌ Erreur: {e}")

Recherche de 'Tevragh Zeina, Nouakchott, Mauritanie'...
✅ Trouvé !
   Latitude: 18.1111443
   Longitude: -16.0038402
   Adresse complète: تفرغ زينة, Nouakchott Ouest نواكشوط الغربية, Mauritanie موريتانيا


In [5]:
print("Début du géocodage des quartiers...")
print("-" * 50)

# Utiliser les quartiers uniques !
quartiers_uniques = df['quartier'].unique()

for quartier in quartiers_uniques:
    print(f"Traitement de: {quartier}")
    
    try:
        # Recherche avec le pays pour plus de précision
        query = f"{quartier}, Nouakchott, Mauritanie"
        location = geocode(query)
        
        if location:
            coord_quartiers[quartier] = {
                'latitude': location.latitude,
                'longitude': location.longitude,
                'adresse': location.address
            }
            print(f"  ✅ Lat: {location.latitude:.4f}, Lon: {location.longitude:.4f}")
        else:
            print(f"  ❌ Non trouvé - on utilisera la table de référence")
            coord_quartiers[quartier] = None
            
    except Exception as e:
        print(f"  ❌ Erreur: {e}")
        coord_quartiers[quartier] = None
    
    print("-" * 50)

# Afficher les résultats
print("\n📊 Récapitulatif du géocodage:")
for quartier, coords in coord_quartiers.items():
    if coords:
        print(f"{quartier}: ({coords['latitude']:.4f}, {coords['longitude']:.4f})")
    else:
        print(f"{quartier}: ⚠️ Non trouvé")

Début du géocodage des quartiers...
--------------------------------------------------
Traitement de: arafat
  ✅ Lat: 18.0460, Lon: -15.9633
--------------------------------------------------
Traitement de: toujounine
  ✅ Lat: 18.0724, Lon: -15.9099
--------------------------------------------------
Traitement de: teyarett
  ✅ Lat: 18.1591, Lon: -15.9272
--------------------------------------------------
Traitement de: tevragh zeina
  ✅ Lat: 18.1111, Lon: -16.0038
--------------------------------------------------
Traitement de: dar naim
  ✅ Lat: 18.1266, Lon: -15.9175
--------------------------------------------------
Traitement de: sebkha
  ✅ Lat: 18.0712, Lon: -16.0026
--------------------------------------------------
Traitement de: riyadh
  ✅ Lat: 18.0107, Lon: -15.9553
--------------------------------------------------
Traitement de: ksar
  ✅ Lat: 18.1049, Lon: -15.9644
--------------------------------------------------

📊 Récapitulatif du géocodage:
arafat: (18.0460, -15.9633)
t

In [ ]:
# Créer les dictionnaires pour la fusion
lat_dict = {}
lon_dict = {}
adresse_dict = {}

for quartier, coords in coord_quartiers.items():
    if coords:  # coords n'est pas None
        lat_dict[quartier] = coords['latitude']
        lon_dict[quartier] = coords['longitude']
        adresse_dict[quartier] = coords['adresse']

# Ajouter les colonnes au DataFrame
df['latitude'] = df['quartier'].map(lat_dict)
df['longitude'] = df['quartier'].map(lon_dict)
df['adresse_osm'] = df['quartier'].map(adresse_dict)

# Vérifier
print("Aperçu des 10 premières lignes:")
print(df[['quartier', 'latitude', 'longitude', 'adresse_osm']].head(10))

# Vérifier qu'il n'y a pas de valeurs manquantes
print(f"\n✅ Lignes avec latitude: {df['latitude'].notna().sum()}/{len(df)}")
print(f"✅ Lignes avec longitude: {df['longitude'].notna().sum()}/{len(df)}")

# Statistiques rapides
print("\n📊 Statistiques par quartier:")
print(df.groupby('quartier').agg({
    'latitude': 'first',
    'longitude': 'first',
}).rename(columns={'prix': 'nb_annonces'}))



Aperçu des 10 premières lignes:
        quartier   latitude  longitude  \
0         arafat  18.045951 -15.963284   
1         arafat  18.045951 -15.963284   
2     toujounine  18.072432 -15.909900   
3         arafat  18.045951 -15.963284   
4       teyarett  18.159145 -15.927189   
5  tevragh zeina  18.111144 -16.003840   
6       dar naim  18.126633 -15.917460   
7  tevragh zeina  18.111144 -16.003840   
8         arafat  18.045951 -15.963284   
9  tevragh zeina  18.111144 -16.003840   

                                         adresse_osm  
0      عرفات, نواكشوط الجنوبية, Mauritanie موريتانيا  
1      عرفات, نواكشوط الجنوبية, Mauritanie موريتانيا  
2  Toujounine توجنين, Nouakchott نواكشوط, توجنين,...  
3      عرفات, نواكشوط الجنوبية, Mauritanie موريتانيا  
4   Teyarett, نواكشوط الشمالية, Mauritanie موريتانيا  
5  تفرغ زينة, Nouakchott Ouest نواكشوط الغربية, M...  
6  Dar Naim المنزل دار نايم, 58, Rue Benichab شار...  
7  تفرغ زينة, Nouakchott Ouest نواكشوط الغربية, M...  
8      عرف

In [ ]:
# Sauvegarder le DataFrame avec les coordonnées
df.to_csv('data_with_coords.csv', index=False)
print("✅ Fichier 'data_with_coords.csv' sauvegardé")

In [ ]:
# Créer une carte centrée sur Nouakchott
carte = folium.Map(location=[18.08, -15.98], zoom_start=11)

# Ajouter un marqueur pour chaque quartier (un seul exemple par quartier)
quartiers_exemples = df.groupby('quartier').first().reset_index()

for _, row in quartiers_exemples.iterrows():
    folium.Marker(
        [row['latitude'], row['longitude']],
        tooltip=row['quartier']
    ).add_to(carte)

# Sauvegarder la carte
# carte.save('carte_quartiers.html')
# print("✅ Carte sauvegardée dans 'carte_quartiers.html'")

# Afficher dans le notebook
carte

In [10]:
# Coordonnées des points stratégiques

# 1. CENTRE-VILLE (Ksar) - On utilise le résultat du géocodage
centre_ville = {
    'nom': 'Ksar',
    'latitude': 18.1049,  # La valeur que tu as obtenue pour Ksar
    'longitude': -15.9644
}

# 2. AÉROPORT (Nouakchott–Oumtounsy International Airport)
# Coordonnées trouvées sur Google Maps / OpenStreetMap
aeroport = {
    'nom': 'Aéroport Oumtounsy',
    'latitude': 18.1092,   # À VÉRIFIER/COMPLÉTER
    'longitude': -15.9483   # À VÉRIFIER/COMPLÉTER
}

# 3. PLAGE - Point de référence (Corniche)
# Choisissons un point central de la côte
plage = {
    'nom': 'Plage de Nouakchott',
    'latitude': 18.0718,   # À VÉRIFIER/COMPLÉTER
    'longitude': -16.0185   # À VÉRIFIER/COMPLÉTER
}

print("Points de référence :")
print(f"📍 Centre-ville (Ksar): ({centre_ville['latitude']:.4f}, {centre_ville['longitude']:.4f})")
print(f"✈️ Aéroport: ({aeroport['latitude']:.4f}, {aeroport['longitude']:.4f})")
print(f"🌊 Plage: ({plage['latitude']:.4f}, {plage['longitude']:.4f})")

Points de référence :
📍 Centre-ville (Ksar): (18.1049, -15.9644)
✈️ Aéroport: (18.1092, -15.9483)
🌊 Plage: (18.0718, -16.0185)


In [11]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    """
    Calcule la distance en kilomètres entre deux points GPS
    """
    # Rayon de la Terre en km
    R = 6371.0
    
    # Conversion en radians
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    
    # Différences
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    # Formule de Haversine
    a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    
    distance = R * c
    return distance

# Test sur Ksar (distance à lui-même doit être 0)
test_distance = haversine(
    centre_ville['latitude'], centre_ville['longitude'],
    centre_ville['latitude'], centre_ville['longitude']
)
print(f"Test: Distance Ksar → Ksar = {test_distance:.2f} km ✅")

Test: Distance Ksar → Ksar = 0.00 km ✅


In [12]:
# Calculer les 3 distances pour chaque ligne
print("Calcul des distances en cours...")

df['dist_centre_ville_km'] = df.apply(
    lambda row: haversine(
        row['latitude'], row['longitude'],
        centre_ville['latitude'], centre_ville['longitude']
    ), axis=1
)

df['dist_aeroport_km'] = df.apply(
    lambda row: haversine(
        row['latitude'], row['longitude'],
        aeroport['latitude'], aeroport['longitude']
    ), axis=1
)

df['dist_plage_km'] = df.apply(
    lambda row: haversine(
        row['latitude'], row['longitude'],
        plage['latitude'], plage['longitude']
    ), axis=1
)

print("✅ Calcul terminé !")

Calcul des distances en cours...
✅ Calcul terminé !


In [14]:
# Afficher un aperçu
print("Aperçu des distances calculées:")
print(df[['quartier', 'dist_centre_ville_km', 'dist_aeroport_km', 'dist_plage_km']].head(10))

# Statistiques par quartier
print("\n📊 Distances moyennes par quartier:")
distances_par_quartier = df.groupby('quartier').agg({
    'dist_centre_ville_km': 'mean',
    'dist_aeroport_km': 'mean',
    'dist_plage_km': 'mean',
}).round(2)

print(distances_par_quartier)

# Vérification rapide : Ksar doit être proche de 0 pour dist_centre_ville
ksar_distance = df[df['quartier'] == 'ksar']['dist_centre_ville_km'].iloc[0]
print(f"\n✅ Distance Ksar → centre-ville: {ksar_distance:.2f} km (devrait être proche de 0)")

Aperçu des distances calculées:
        quartier  dist_centre_ville_km  dist_aeroport_km  dist_plage_km
0         arafat              6.555870          7.209089       6.506583
1         arafat              6.555870          7.209089       6.506583
2     toujounine              6.798401          5.760955      11.480216
3         arafat              6.555870          7.209089       6.506583
4       teyarett              7.200247          5.984872      13.691263
5  tevragh zeina              4.225777          5.873823       4.641186
6       dar naim              5.518094          3.792139      12.297215
7  tevragh zeina              4.225777          5.873823       4.641186
8         arafat              6.555870          7.209089       6.506583
9  tevragh zeina              4.225777          5.873823       4.641186

📊 Distances moyennes par quartier:
               dist_centre_ville_km  dist_aeroport_km  dist_plage_km
quartier                                                            
ar

In [15]:
# Sauvegarder le DataFrame avec les nouvelles distances
df.to_csv('data_test_with_distances.csv', index=False)
print("✅ Fichier 'data_test_with_distances.csv' sauvegardé")

✅ Fichier 'data_test_with_distances.csv' sauvegardé


In [ ]:
import overpy
import time
import pandas as pd
import numpy as np
from tqdm import tqdm

print("=" * 70)
print("🚀 CRÉATION DES GROUPES GPS POUR LE FICHIER TEST")
print("=" * 70)

# Recharger les données du fichier test avec distances
df_test = pd.read_csv('data_test_with_distances.csv') 
print(f"✅ Données chargées: {len(df_test)} biens")
print(f"📋 Colonnes disponibles: {df_test.columns.tolist()}")

# Créer des groupes de biens très proches (arrondi à 4 décimales ≈ 10 mètres)
df_test['lat_round'] = df_test['latitude'].round(4)
df_test['lon_round'] = df_test['longitude'].round(4)
df_test['groupe_gps'] = df_test['lat_round'].astype(str) + '_' + df_test['lon_round'].astype(str)

print(f"\n📍 Nombre de groupes GPS uniques (précision ~10m): {df_test['groupe_gps'].nunique()}")

# Afficher les groupes (sans utiliser 'prix')
groupes = df_test.groupby('groupe_gps').agg({
    'latitude': 'first',
    'longitude': 'first',
    'lat_round': 'first',
    'lon_round': 'first',
    'quartier': 'first',
    'id': 'count'  # Utiliser 'id' au lieu de 'prix' pour compter
}).rename(columns={'id': 'nb_biens'}).reset_index()

print("\n📊 Répartition des groupes:")
print(groupes[['groupe_gps', 'quartier', 'nb_biens']].head(10))
print(f"\n📈 Nombre total de groupes à traiter: {len(groupes)}")


🚀 CRÉATION DES GROUPES GPS POUR LE FICHIER TEST
✅ Données chargées: 289 biens
📋 Colonnes disponibles: ['id', 'titre', 'surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb', 'quartier', 'description', 'caracteristiques', 'source', 'date_publication', 'quartier_encoded', 'latitude', 'longitude', 'adresse_osm', 'dist_centre_ville_km', 'dist_aeroport_km', 'dist_plage_km']

📍 Nombre de groupes GPS uniques (précision ~10m): 8

📊 Répartition des groupes:
         groupe_gps       quartier  nb_biens
0  18.0107_-15.9553         riyadh         3
1   18.046_-15.9633         arafat        61
2  18.0712_-16.0026         sebkha         3
3  18.0724_-15.9099     toujounine        29
4  18.1049_-15.9644           ksar        12
5  18.1111_-16.0038  tevragh zeina        93
6  18.1266_-15.9175       dar naim        21
7  18.1591_-15.9272       teyarett        67

📈 Nombre total de groupes à traiter: 8
✅ Groupes GPS sauvegardés dans 'groupes_gps_test.csv'


In [49]:
# Version avec gestion des erreurs et réessais automatiques
import time
import random

def count_pois_around_robust(lat, lon, rayon=1000, max_retries=3):
    """
    Version robuste avec réessais en cas d'erreur
    """
    resultats = {
        'ecoles': 0,
        'mosquees': 0,
        'commerces': 0,
        'hopitaux': 0,
        'total': 0
    }
    
    # Types de POIs à chercher
    poi_queries = [
        ('ecoles', """
            (
              node["amenity"="school"](around:{rayon},{lat},{lon});
              way["amenity"="school"](around:{rayon},{lat},{lon});
            );
        """),
        ('mosquees', """
            (
              node["amenity"="place_of_worship"]["religion"="muslim"](around:{rayon},{lat},{lon});
              way["amenity"="place_of_worship"]["religion"="muslim"](around:{rayon},{lat},{lon});
            );
        """),
        ('commerces', """
            (
              node["shop"](around:{rayon},{lat},{lon});
              way["shop"](around:{rayon},{lat},{lon});
            );
        """),
        ('hopitaux', """
            (
              node["amenity"="hospital"](around:{rayon},{lat},{lon});
              way["amenity"="hospital"](around:{rayon},{lat},{lon});
              node["amenity"="clinic"](around:{rayon},{lat},{lon});
              way["amenity"="clinic"](around:{rayon},{lat},{lon});
            );
        """)
    ]
    
    for poi_name, poi_query_template in poi_queries:
        for attempt in range(max_retries):
            try:
                # Construire la requête complète
                query = f"""
                [out:json];
                {poi_query_template}
                out body;
                """
                query = query.format(rayon=rayon, lat=lat, lon=lon)
                
                # Envoyer la requête
                result = api.query(query)
                count = len(result.nodes) + len(result.ways)
                resultats[poi_name] = count
                
                print(f"    ✅ {poi_name}: {count}")
                break  # Sortir de la boucle de réessai si réussi
                
            except Exception as e:
                if attempt < max_retries - 1:
                    # Attendre avant de réessayer (temps exponentiel)
                    wait_time = (attempt + 1) * 5 + random.randint(1, 5)
                    print(f"    ⚠️ {poi_name}: erreur (essai {attempt+1}/{max_retries}), nouvel essai dans {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"    ❌ {poi_name}: échec après {max_retries} essais")
        
        # Pause entre les types de POIs
        time.sleep(3)
    
    # Calcul du total
    resultats['total'] = (resultats['ecoles'] + resultats['mosquees'] + 
                         resultats['commerces'] + resultats['hopitaux'])
    
    return resultats

# Test avec la version robuste
print("🔍 Test sur Ksar (avec réessais):")
test = count_pois_around_robust(18.1049, -15.9644)
print(f"\n📊 Résultats finaux:")
print(f"  🏫 Écoles: {test['ecoles']}")
print(f"  🕌 Mosquées: {test['mosquees']}")
print(f"  🏪 Commerces: {test['commerces']}")
print(f"  🏥 Hôpitaux: {test['hopitaux']}")
print(f"  📊 Total: {test['total']}")

🔍 Test sur Ksar (avec réessais):
    ✅ ecoles: 6
    ✅ mosquees: 3
    ✅ commerces: 60
    ✅ hopitaux: 5

📊 Résultats finaux:
  🏫 Écoles: 6
  🕌 Mosquées: 3
  🏪 Commerces: 60
  🏥 Hôpitaux: 5
  📊 Total: 74


In [61]:
# ============================================
# TRAITEMENT D'UN QUARTIER À LA FOIS
# ============================================

import overpy
import time
import random
import pandas as pd
import os

# Connexion à l'API
api = overpy.Overpass()

def count_pois_around_robust(lat, lon, rayon=1000, max_retries=3):
    """
    Version robuste avec réessais en cas d'erreur
    """
    resultats = {
        'ecoles': 0,
        'mosquees': 0,
        'commerces': 0,
        'hopitaux': 0,
        'total': 0
    }
    
    # Types de POIs à chercher
    poi_queries = [
        ('ecoles', """
            (
              node["amenity"="school"](around:{rayon},{lat},{lon});
              way["amenity"="school"](around:{rayon},{lat},{lon});
            );
        """),
        ('mosquees', """
            (
              node["amenity"="place_of_worship"]["religion"="muslim"](around:{rayon},{lat},{lon});
              way["amenity"="place_of_worship"]["religion"="muslim"](around:{rayon},{lat},{lon});
            );
        """),
        ('commerces', """
            (
              node["shop"](around:{rayon},{lat},{lon});
              way["shop"](around:{rayon},{lat},{lon});
            );
        """),
        ('hopitaux', """
            (
              node["amenity"="hospital"](around:{rayon},{lat},{lon});
              way["amenity"="hospital"](around:{rayon},{lat},{lon});
              node["amenity"="clinic"](around:{rayon},{lat},{lon});
              way["amenity"="clinic"](around:{rayon},{lat},{lon});
            );
        """)
    ]
    
    for poi_name, poi_query_template in poi_queries:
        for attempt in range(max_retries):
            try:
                # Construire la requête complète
                query = f"""
                [out:json];
                {poi_query_template}
                out body;
                """
                query = query.format(rayon=rayon, lat=lat, lon=lon)
                
                # Envoyer la requête
                result = api.query(query)
                count = len(result.nodes) + len(result.ways)
                resultats[poi_name] = count
                
                print(f"    ✅ {poi_name}: {count}")
                break  # Sortir de la boucle de réessai si réussi
                
            except Exception as e:
                if attempt < max_retries - 1:
                    # Attendre avant de réessayer (temps exponentiel)
                    wait_time = (attempt + 1) * 5 + random.randint(1, 5)
                    print(f"    ⚠️ {poi_name}: erreur (essai {attempt+1}/{max_retries}), nouvel essai dans {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"    ❌ {poi_name}: échec après {max_retries} essais")
        
        # Pause entre les types de POIs
        time.sleep(3)
    
    # Calcul du total
    resultats['total'] = (resultats['ecoles'] + resultats['mosquees'] + 
                         resultats['commerces'] + resultats['hopitaux'])
    
    return resultats

def sauvegarder_resultat(quartier, groupe_gps, lat, lon, resultats, fichier="poi_quartiers.csv"):
    """
    Sauvegarde ou ajoute le résultat d'un quartier dans le fichier CSV
    """
    # Créer un dictionnaire avec les résultats
    nouveau = {
        'groupe_gps': groupe_gps,
        'quartier': quartier,
        'latitude': lat,
        'longitude': lon,
        'nb_ecoles_1km': resultats['ecoles'],
        'nb_mosquees_1km': resultats['mosquees'],
        'nb_commerce_1km': resultats['commerces'],
        'nb_hopitaux_1km': resultats['hopitaux'],
        'nb_total_pois_1km': resultats['total']
    }
    
    # Vérifier si le fichier existe déjà
    if os.path.exists(fichier):
        # Charger le fichier existant
        df_existant = pd.read_csv(fichier)
        
        # Vérifier si ce quartier est déjà traité
        if groupe_gps in df_existant['groupe_gps'].values:
            # Mettre à jour la ligne existante
            df_existant.loc[df_existant['groupe_gps'] == groupe_gps, 
                           ['nb_ecoles_1km', 'nb_mosquees_1km', 'nb_commerce_1km', 
                            'nb_hopitaux_1km', 'nb_total_pois_1km']] = [
                                resultats['ecoles'], resultats['mosquees'], 
                                resultats['commerces'], resultats['hopitaux'], 
                                resultats['total']
                            ]
            df_existant.to_csv(fichier, index=False)
            print(f"   ✅ Quartier mis à jour dans {fichier}")
        else:
            # Ajouter la nouvelle ligne
            df_nouveau = pd.DataFrame([nouveau])
            df_combine = pd.concat([df_existant, df_nouveau], ignore_index=True)
            df_combine.to_csv(fichier, index=False)
            print(f"   ✅ Quartier ajouté à {fichier}")
    else:
        # Créer un nouveau fichier
        df_nouveau = pd.DataFrame([nouveau])
        df_nouveau.to_csv(fichier, index=False)
        print(f"   ✅ Fichier {fichier} créé avec le premier quartier")

# ============================================
# TRAITER UN SEUL QUARTIER À LA FOIS
# ============================================

# Liste des quartiers avec leurs coordonnées
quartiers = [
    {'nom': 'Riyadh', 'groupe': '18.0107_-15.9553', 'lat': 18.0107, 'lon': -15.9553, 'biens': 13},
    {'nom': 'Arafat', 'groupe': '18.046_-15.9633', 'lat': 18.046, 'lon': -15.9633, 'biens': 244},
    {'nom': 'Sebkha', 'groupe': '18.0712_-16.0026', 'lat': 18.0712, 'lon': -16.0026, 'biens': 10},
    {'nom': 'Toujounine', 'groupe': '18.0724_-15.9099', 'lat': 18.0724, 'lon': -15.9099, 'biens': 115},
    {'nom': 'Ksar', 'groupe': '18.1049_-15.9644', 'lat': 18.1049, 'lon': -15.9644, 'biens': 46},
    {'nom': 'Tevragh Zeina', 'groupe': '18.1111_-16.0038', 'lat': 18.1111, 'lon': -16.0038, 'biens': 373},
    {'nom': 'Dar Naim', 'groupe': '18.1266_-15.9175', 'lat': 18.1266, 'lon': -15.9175, 'biens': 82},
    {'nom': 'Teyarett', 'groupe': '18.1591_-15.9272', 'lat': 18.1591, 'lon': -15.9272, 'biens': 270},
]

# CHOISIS LE QUARTIER À TRAITER (change l'index)
index_quartier = 7  # 0=Riyadh, 1=Arafat, 2=Sebkha, 3=Toujounine, 4=Ksar, 5=Tevragh Zeina, 6=Dar Naim, 7=Teyarett

quartier = quartiers[index_quartier]

print("=" * 70)
print(f"📍 TRAITEMENT DE {quartier['nom']}")
print("=" * 70)
print(f"   Groupe GPS: {quartier['groupe']}")
print(f"   Latitude: {quartier['lat']}, Longitude: {quartier['lon']}")
print(f"   Nombre de biens dans ce groupe: {quartier['biens']}")
print("-" * 70)

# Traiter le quartier
resultats = count_pois_around_robust(
    lat=quartier['lat'],
    lon=quartier['lon'],
    max_retries=5  # Plus de tentatives pour être sûr
)

print(f"\n📊 RÉSULTATS POUR {quartier['nom']}:")
print(f"  🏫 Écoles: {resultats['ecoles']}")
print(f"  🕌 Mosquées: {resultats['mosquees']}")
print(f"  🏪 Commerces: {resultats['commerces']}")
print(f"  🏥 Hôpitaux: {resultats['hopitaux']}")
print(f"  📊 Total: {resultats['total']}")
print("-" * 70)

# Sauvegarder dans le fichier
sauvegarder_resultat(
    quartier=quartier['nom'],
    groupe_gps=quartier['groupe'],
    lat=quartier['lat'],
    lon=quartier['lon'],
    resultats=resultats,
    fichier="poi_quartiers_complets.csv"
)

print("\n✅ Traitement terminé pour ce quartier !")
# print("Change l'index_quartier pour traiter le suivant")

📍 TRAITEMENT DE Teyarett
   Groupe GPS: 18.1591_-15.9272
   Latitude: 18.1591, Longitude: -15.9272
   Nombre de biens dans ce groupe: 270
----------------------------------------------------------------------
    ⚠️ ecoles: erreur (essai 1/5), nouvel essai dans 8s...
    ✅ ecoles: 0
    ✅ mosquees: 0
    ⚠️ commerces: erreur (essai 1/5), nouvel essai dans 8s...
    ✅ commerces: 0
    ✅ hopitaux: 0

📊 RÉSULTATS POUR Teyarett:
  🏫 Écoles: 0
  🕌 Mosquées: 0
  🏪 Commerces: 0
  🏥 Hôpitaux: 0
  📊 Total: 0
----------------------------------------------------------------------
   ✅ Quartier ajouté à poi_quartiers_complets.csv

✅ Traitement terminé pour ce quartier !


In [19]:
# ============================================
# FUSION CORRIGÉE AVEC UNIFORMISATION DE LA CASSE
# ============================================

import pandas as pd

print("=" * 70)
print("🚀 FUSION CORRIGÉE AVEC UNIFORMISATION DE LA CASSE")
print("=" * 70)

# 1. Charger les fichiers
file1 = "/home/fatma/Documents/machinelearning/capstone_project/geo_enrichissement/data_test_with_distances.csv"
file2 = "/home/fatma/Documents/machinelearning/capstone_project/geo_enrichissement/poi_quartiers_complets.csv"

df_principal = pd.read_csv(file1)
df_poi = pd.read_csv(file2)

print(f"\n📂 data_test_with_distances.csv: {df_principal.shape}")
print(f"📂 poi_quartiers_complets.csv: {df_poi.shape}")

# 2. Uniformiser la casse (tout en minuscules)
df_principal['quartier_key'] = df_principal['quartier'].str.lower().str.strip()
df_poi['quartier_key'] = df_poi['quartier'].str.lower().str.strip()

print("\n🔍 Vérification après uniformisation:")
print(f"   Quartiers dans principal: {sorted(df_principal['quartier_key'].unique())}")
print(f"   Quartiers dans POIs: {sorted(df_poi['quartier_key'].unique())}")

# 3. Définir les colonnes POIs
colonnes_poi = ['nb_ecoles_1km', 'nb_mosquees_1km', 'nb_commerce_1km', 
                'nb_hopitaux_1km', 'nb_total_pois_1km']

# 4. Fusionner
print("\n🔗 Fusion des données...")
df_final = df_principal.merge(
    df_poi[['quartier_key'] + colonnes_poi],
    left_on='quartier_key',
    right_on='quartier_key',
    how='left'
)

print("   ✅ Fusion terminée")

# 5. Vérifier les résultats
print("\n📊 Aperçu des résultats (10 premiers):")
print(df_final[['quartier', 'nb_ecoles_1km', 'nb_mosquees_1km', 
                'nb_commerce_1km', 'nb_hopitaux_1km', 'nb_total_pois_1km']].head(10))

# 6. Vérifier les valeurs manquantes
print("\n🔍 Valeurs manquantes:")
print(df_final[colonnes_poi].isnull().sum())

# 7. Supprimer la colonne temporaire
df_final = df_final.drop('quartier_key', axis=1)

# 8. Sauvegarder
output_file = "/home/fatma/Documents/machinelearning/capstone_project/geo_enrichissement/data_test_final.csv"
df_final.to_csv(output_file, index=False)
print(f"\n✅ Fichier sauvegardé: {output_file}")

# 9. Vérification par quartier
print("\n📈 Vérification par quartier (première occurrence):")
verif = df_final.groupby('quartier')[colonnes_poi].first()
print(verif)

print("\n🎉 FUSION TERMINÉE AVEC SUCCÈS !")

🚀 FUSION CORRIGÉE AVEC UNIFORMISATION DE LA CASSE

📂 data_test_with_distances.csv: (289, 18)
📂 poi_quartiers_complets.csv: (8, 9)

🔍 Vérification après uniformisation:
   Quartiers dans principal: ['arafat', 'dar naim', 'ksar', 'riyadh', 'sebkha', 'tevragh zeina', 'teyarett', 'toujounine']
   Quartiers dans POIs: ['arafat', 'dar naim', 'ksar', 'riyadh', 'sebkha', 'tevragh zeina', 'teyarett', 'toujounine']

🔗 Fusion des données...
   ✅ Fusion terminée

📊 Aperçu des résultats (10 premiers):
        quartier  nb_ecoles_1km  nb_mosquees_1km  nb_commerce_1km  \
0         arafat             14               21               23   
1         arafat             14               21               23   
2     toujounine              6                5                4   
3         arafat             14               21               23   
4       teyarett              0                0                0   
5  tevragh zeina              3                3               16   
6       dar naim       